# ANEEL RAG — Módulo 5: Embedding e Indexação (v2)

Notebook **standalone** — todo o código está embutido aqui, sem `git clone`.

## Antes de rodar
1. Ambiente de execução → Alterar tipo de ambiente de execução → **GPU T4**
2. Confirme que seu Google Drive tem:
   - `Meu Drive/ANEEL-RAG/chunks.parquet` (260 MB)

## Rode as células em ordem
A **Célula 7** é a principal — embeda e indexa tudo. Vai demorar ~3h para os 228k chunks.

In [ ]:
# CÉLULA 1 — Verifica GPU
# Deve mostrar Tesla T4. Se mostrar 'No GPU', vá em:
# Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4
!nvidia-smi

In [ ]:
# CÉLULA 2 — Monta o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CÉLULA 3 — Instala dependências com versões fixas
# Torch com CUDA já vem no Colab — só instalamos o restante
!pip install -q \
    FlagEmbedding==1.4.0 \
    qdrant-client==1.17.1 \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    sentencepiece==0.2.0 \
    tqdm
print('✓ Dependências instaladas')

In [ ]:
# CÉLULA 4 — Verifica torch + CUDA
import torch
print(f'Torch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('ATENÇÃO: sem GPU — o embedding será muito lento!')

In [ ]:
# CÉLULA 5 — Copia chunks.parquet do Drive para disco local
# Disco local do Colab é SSD rápido — muito melhor que Drive FUSE
import os, shutil

src = '/content/drive/MyDrive/ANEEL-RAG/chunks.parquet'
dst = '/content/chunks.parquet'

if not os.path.exists(src):
    raise FileNotFoundError(
        f'Não encontrado: {src}\n'
        'Verifique se chunks.parquet está em: Meu Drive/ANEEL-RAG/chunks.parquet'
    )

shutil.copy(src, dst)
size_mb = os.path.getsize(dst) / 1e6
print(f'✓ chunks.parquet copiado ({size_mb:.0f} MB)')

if size_mb < 200:
    print('AVISO: arquivo menor que 200 MB — pode estar incompleto!')
    print('O arquivo completo deve ter ~260 MB.')

In [ ]:
# CÉLULA 6 — CONFIGURAÇÃO
# =============================================================
#  LEIA ANTES DE RODAR A CÉLULA 7
# =============================================================
#
#  MODO = 'COMPLETO'
#    → Começa do zero com todos os 228k chunks (~3h na T4)
#    → Produz um banco de dados limpo e completo
#    → RECOMENDADO se você tem créditos suficientes
#
#  MODO = 'RETOMAR'
#    → Indexa somente os chunks restantes (~1.5h na T4)
#    → Salva em qdrant_db_new (banco separado do anterior)
#    → Use se quiser manter os 124k chunks já indexados
#    → O módulo 6 buscará nos dois bancos automaticamente
#
MODO = 'COMPLETO'    # <── 'COMPLETO' ou 'RETOMAR'
SKIP_FIRST = 124633  # só usado quando MODO = 'RETOMAR'
# =============================================================
print(f'Modo selecionado: {MODO}')
if MODO == 'RETOMAR':
    print(f'Pulará os primeiros {SKIP_FIRST:,} chunks')
    print('Atenção: o banco antigo (qdrant_db) NÃO será aberto — sem risco de crash')

In [ ]:
# CÉLULA 7 — Embedding e Indexação
# Todo o código está aqui — sem dependência do repositório git

import gc, hashlib, json, math, os, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, HnswConfigDiff, OptimizersConfigDiff,
    PayloadSchemaType, PointStruct,
    SparseIndexParams, SparseVectorParams, SparseVector, VectorParams,
)

# ── Parâmetros ────────────────────────────────────────────────────────────────
CHUNKS_PATH     = '/content/chunks.parquet'
DRIVE_DIR       = '/content/drive/MyDrive/ANEEL-RAG'
COLLECTION      = 'aneel_legislacao'
EMBED_MODEL     = 'BAAI/bge-m3'
BATCH_SIZE      = 128    # T4 16 GB comporta 128 com fp16
UPSERT_BATCH    = 1000   # pontos por upsert (menos chamadas I/O)
CHECKPOINT_N    = 50_000 # salva zip no Drive a cada N chunks indexados
LOG_EVERY       = 500    # loga progresso a cada N chunks
DENSE_DIM       = 1024
HNSW_M          = 16
HNSW_EF         = 200
INDEXABLE_TYPES = {'child', 'ementa', 'fallback'}

# Caminho local do Qdrant (nunca Drive durante ingestão — evita degradação FUSE)
if MODO == 'COMPLETO':
    QDRANT_LOCAL = '/content/qdrant_db'
    SKIP = 0
else:
    QDRANT_LOCAL = '/content/qdrant_db_new'
    SKIP = SKIP_FIRST

print(f'=== CONFIG ===')
print(f'Modo:        {MODO}')
print(f'Qdrant:      {QDRANT_LOCAL}')
print(f'Skip first:  {SKIP:,}')
print(f'Batch embed: {BATCH_SIZE}')
print(f'Batch upsert:{UPSERT_BATCH}')

# ── Funções auxiliares ────────────────────────────────────────────────────────

def chunk_id_to_int(chunk_id: str) -> int:
    digest = hashlib.md5(chunk_id.encode()).digest()
    return int.from_bytes(digest[:8], 'big')

def sparse_dict_to_qdrant(d: dict) -> SparseVector:
    if not d:
        return SparseVector(indices=[], values=[])
    return SparseVector(
        indices=[int(k) for k in d],
        values=[float(v) for v in d.values()],
    )

def build_points(chunks, dense_vecs, sparse_vecs):
    points = []
    for i, chunk in enumerate(chunks):
        payload = {k: v for k, v in chunk.items()}
        payload['is_ementa'] = (chunk.get('chunk_type') == 'ementa')
        for k, v in payload.items():
            if isinstance(v, float) and math.isnan(v):
                payload[k] = None
        points.append(PointStruct(
            id=chunk_id_to_int(chunk.get('chunk_id', '')),
            vector={
                'dense':  dense_vecs[i].tolist(),
                'sparse': sparse_dict_to_qdrant(sparse_vecs[i]),
            },
            payload=payload,
        ))
    return points

def upload_with_retry(client, points, retries=3):
    delay = 1.0
    for attempt in range(retries):
        try:
            client.upsert(collection_name=COLLECTION, points=points)
            return
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f'  upsert falhou ({e}), tentando em {delay:.0f}s...')
            time.sleep(delay)
            delay *= 2

def checkpoint_to_drive(n_indexed):
    """Zipa qdrant_db local e salva no Drive (atomicamente via .tmp)."""
    zip_path = f'{DRIVE_DIR}/qdrant_checkpoint_{n_indexed:07d}.zip'
    tmp_path = zip_path + '.tmp'
    print(f'\n[CHECKPOINT {n_indexed:,}] Zipando para Drive...')
    t0 = time.time()
    with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_STORED) as zf:
        for fp in Path(QDRANT_LOCAL).rglob('*'):
            if fp.is_file():
                zf.write(fp, fp.relative_to(QDRANT_LOCAL))
    os.rename(tmp_path, zip_path)
    mb = os.path.getsize(zip_path) / 1e6
    print(f'[CHECKPOINT] ✓ {mb:.0f} MB em {time.time()-t0:.0f}s → {Path(zip_path).name}')

# ── 1. Carregar e filtrar chunks ──────────────────────────────────────────────
print('\n=== 1. Carregando chunks ===')
df = pd.read_parquet(CHUNKS_PATH)
print(f'Total no parquet: {len(df):,}')
print(df['chunk_type'].value_counts().to_string())

df = df[df['chunk_type'].isin(INDEXABLE_TYPES)].copy()
print(f'Indexáveis (child+ementa+fallback): {len(df):,}')

mask_vazio = df['texto'].isna() | (df['texto'].str.strip() == '')
n_vazios = mask_vazio.sum()
if n_vazios:
    print(f'Chunks sem texto (PDFs indisponíveis): {n_vazios:,} — ignorados')
df = df[~mask_vazio].copy()
print(f'Com texto válido: {len(df):,}')

if SKIP > 0:
    df = df.iloc[SKIP:].copy()
    print(f'Após skip-first {SKIP:,}: {len(df):,} restantes')

records = df.to_dict('records')
texts   = [r['texto'] for r in records]
total   = len(records)
print(f'\nChunks a indexar nesta sessão: {total:,}')

del df
gc.collect()

# ── 2. Criar collection Qdrant (banco LOCAL) ──────────────────────────────────
print('\n=== 2. Configurando Qdrant (banco LOCAL) ===')
os.makedirs(QDRANT_LOCAL, exist_ok=True)
client = QdrantClient(path=QDRANT_LOCAL)

existing = {c.name for c in client.get_collections().collections}
if COLLECTION in existing:
    client.delete_collection(COLLECTION)
    print(f"Collection '{COLLECTION}' deletada (reset).")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={'dense': VectorParams(size=DENSE_DIM, distance=Distance.COSINE)},
    sparse_vectors_config={'sparse': SparseVectorParams(index=SparseIndexParams(on_disk=False))},
    hnsw_config=HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF),
    optimizers_config=OptimizersConfigDiff(
        indexing_threshold=0,        # pausa HNSW durante ingestão (muito mais rápido)
        memmap_threshold=20_000,
    ),
)
for field, schema in {
    'tipo_sigla': PayloadSchemaType.KEYWORD,
    'ano':        PayloadSchemaType.INTEGER,
    'chunk_type': PayloadSchemaType.KEYWORD,
    'is_ementa':  PayloadSchemaType.BOOL,
    'revogada':   PayloadSchemaType.BOOL,
}.items():
    client.create_payload_index(COLLECTION, field, schema)
print(f"Collection '{COLLECTION}' criada com HNSW pausado durante ingestão.")

# Fecha client antes de carregar o modelo (libera RAM antes do pico)
client.close()
del client
gc.collect()

# ── 3. Carregar modelo ────────────────────────────────────────────────────────
print('\n=== 3. Carregando BAAI/bge-m3 (~4 GB, pode demorar na 1ª vez) ===')
from FlagEmbedding import BGEM3FlagModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = BGEM3FlagModel(EMBED_MODEL, use_fp16=(device == 'cuda'), device=device)
print('✓ Modelo carregado!')

# Reabre Qdrant após modelo pronto
client = QdrantClient(path=QDRANT_LOCAL)
# Garante optimizer pausado (precaução)
client.update_collection(
    COLLECTION,
    optimizer_config=OptimizersConfigDiff(indexing_threshold=0),
)

# ── 4. Embedding + Indexação ──────────────────────────────────────────────────
print(f'\n=== 4. Embedding + Indexação ({total:,} chunks) ===')
print(f'Estimativa: ~{total/1200:.0f} min na T4 a 1200 chunks/min')
print()

start_time  = time.time()
n_uploaded  = 0
pending     = []
last_log    = 0
last_chkpt  = 0

def log_progress(n_done):
    elapsed = time.time() - start_time
    rate    = n_done / (elapsed / 60) if elapsed > 0 else 0
    eta     = (total - n_done) / rate if rate > 0 else float('inf')
    print(
        f"[{time.strftime('%H:%M:%S')}] "
        f"{n_done:>7,}/{total:,}  "
        f"{rate:.0f} chunks/min  "
        f"ETA {eta:.0f} min"
    )

def flush(force=False):
    global n_uploaded, pending
    if pending and (force or len(pending) >= UPSERT_BATCH):
        upload_with_retry(client, pending)
        n_uploaded += len(pending)
        pending = []

for i in range(0, total, BATCH_SIZE):
    batch_chunks = records[i : i + BATCH_SIZE]
    batch_texts  = texts[i : i + BATCH_SIZE]

    try:
        out = model.encode(
            batch_texts,
            batch_size=BATCH_SIZE,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        )
        pts = build_points(batch_chunks, out['dense_vecs'], out['lexical_weights'])
        pending.extend(pts)
        flush()
    except Exception as e:
        print(f'ERRO batch {i}–{i+BATCH_SIZE}: {e}')
        continue

    n_done = i + len(batch_chunks)

    if n_done - last_log >= LOG_EVERY:
        log_progress(n_done)
        last_log = n_done

    if n_done - last_chkpt >= CHECKPOINT_N:
        flush(force=True)
        checkpoint_to_drive(SKIP + n_done)
        last_chkpt = n_done

flush(force=True)

elapsed = time.time() - start_time
print(f'\n✓ Ingestão concluída: {n_uploaded:,} pontos em {elapsed/60:.1f} min')

# ── 5. Reabilitar HNSW e aguardar otimização ──────────────────────────────────
print('\n=== 5. Reabilitando HNSW (otimização de ~10-20 min) ===')
client.update_collection(
    COLLECTION,
    optimizer_config=OptimizersConfigDiff(indexing_threshold=20_000),
)
print('Optimizer ativado. Aguardando 15 min para consolidar segmentos...')
for i in range(15):
    time.sleep(60)
    info = client.get_collection(COLLECTION)
    count = getattr(info, 'points_count', None) or getattr(info, 'vectors_count', None) or 0
    print(f'  {i+1:2d}/15 min — {count:,} pontos confirmados')

print('✓ Otimização concluída (ou em andamento — OK para salvar)')

# ── 6. Salvar estatísticas ────────────────────────────────────────────────────
info  = client.get_collection(COLLECTION)
count = getattr(info, 'points_count', None) or getattr(info, 'vectors_count', None) or 0
stats = {
    'indexed_chunks':        count,
    'collection':            COLLECTION,
    'qdrant_path':           QDRANT_LOCAL,
    'indexing_time_minutes': round(elapsed / 60, 2),
    'skip_first':            SKIP,
    'modo':                  MODO,
}
Path('/content/index_stats.json').write_text(json.dumps(stats, indent=2, ensure_ascii=False))
print(f'Stats: {stats}')
print('\n→ Rode a CÉLULA 8 para salvar o qdrant_db no Drive!')

In [ ]:
# CÉLULA 8 — Salva o qdrant_db no Drive
# Rode assim que a Célula 7 terminar.
# Gera qdrant_db_final.zip no seu Drive.

import os, time, zipfile
from pathlib import Path

qdrant_local = QDRANT_LOCAL   # definido na Célula 7
zip_path     = '/content/drive/MyDrive/ANEEL-RAG/qdrant_db_final.zip'
tmp_path     = zip_path + '.tmp'

print(f'Comprimindo {qdrant_local} → Drive...')
t0 = time.time()

with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_STORED) as zf:
    for fp in Path(qdrant_local).rglob('*'):
        if fp.is_file():
            zf.write(fp, fp.relative_to(qdrant_local))

os.rename(tmp_path, zip_path)
size_mb = os.path.getsize(zip_path) / 1e6
elapsed = time.time() - t0

print(f'✓ Salvo em qdrant_db_final.zip ({size_mb:.0f} MB, {elapsed:.0f}s)')
print('Agora baixe o arquivo do Google Drive para seu PC.')

In [ ]:
# CÉLULA 9 — (OPCIONAL) Salva index_stats.json no Drive
import shutil
shutil.copy('/content/index_stats.json', '/content/drive/MyDrive/ANEEL-RAG/index_stats.json')
print('✓ index_stats.json salvo no Drive')